In [2]:
import datasets
import numpy as np
import json
from pathlib import Path

# Load a HuggingFace dataset from disk
dataset_path = Path("data/hf_dataset/CPCB/H")
ds = datasets.load_from_disk(str(dataset_path)).with_format("numpy")

print(f"Dataset: {ds}")
print(f"Number of series: {len(ds)}")
print(f"Columns: {ds.column_names}")
print(f"First entry keys: {ds[0].keys()}")
print(f"First entry target shape: {ds[0]['target'].shape}")

Dataset: Dataset({
    features: ['item_id', 'start', 'freq', 'target'],
    num_rows: 1104
})
Number of series: 1104
Columns: ['item_id', 'start', 'freq', 'target']
First entry keys: dict_keys(['item_id', 'start', 'freq', 'target'])
First entry target shape: (26304,)


In [3]:
ds['item_id'][0]

np.str_('site_105_North_Campus_DU_Delhi_IMD_CO')

In [4]:
ds['start'][0]

array('2022-07-01T00:00:00', dtype='datetime64[s]')

In [4]:
# Load predictions.npz
results_dir = Path("output/results/chronos_bolt_base/CPCB/H/short")

with open(results_dir / "config.json") as f:
    cfg = json.load(f)

preds_data = np.load(results_dir / "predictions.npz")
preds = preds_data["predictions_quantiles"].astype(float) * cfg["prediction_scale_factor"]
quantile_levels = np.array(cfg["quantile_levels"])
timestamps = preds_data["timestamps"]

print(f"Config: {json.dumps({k: v for k, v in cfg.items() if k != 'item_ids'}, indent=2)}")
print(f"\nPredictions shape: {preds.shape}  (num_series, num_windows, num_quantiles, num_variates, prediction_length)")
print(f"Quantile levels: {quantile_levels}")
print(f"Timestamps shape: {timestamps.shape}")
print(f"Scale factor: {cfg['prediction_scale_factor']}")

Config: {
  "dataset_config": "CPCB/H/short",
  "num_series": 1104,
  "num_windows": 60,
  "num_variates": 1,
  "prediction_length": 24,
  "num_quantiles": 9,
  "quantile_levels": [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9
  ],
  "freq": "h",
  "seasonality": 24,
  "context_length": 168,
  "metric_names": [
    "MSE",
    "MAE",
    "RMSE",
    "MAPE",
    "sMAPE",
    "MASE",
    "ND",
    "CRPS"
  ],
  "prediction_scale_factor": 1.0,
  "model": "chronos-bolt-base"
}

Predictions shape: (1104, 60, 9, 1, 24)  (num_series, num_windows, num_quantiles, num_variates, prediction_length)
Quantile levels: [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]
Timestamps shape: (1104, 60, 24)
Scale factor: 1.0


In [8]:
import pandas as pd
pd.to_datetime(timestamps[0], unit='s')

DatetimeIndex(['<DatetimeArray>
['2025-06-30 00:00:00', '2025-06-30 01:00:00', '2025-06-30 02:00:00',
 '2025-06-30 03:00:00', '2025-06-30 04:00:00', '2025-06-30 05:00:00',
 '2025-06-30 06:00:00', '2025-06-30 07:00:00', '2025-06-30 08:00:00',
 '2025-06-30 09:00:00', '2025-06-30 10:00:00', '2025-06-30 11:00:00',
 '2025-06-30 12:00:00', '2025-06-30 13:00:00', '2025-06-30 14:00:00',
 '2025-06-30 15:00:00', '2025-06-30 16:00:00', '2025-06-30 17:00:00',
 '2025-06-30 18:00:00', '2025-06-30 19:00:00', '2025-06-30 20:00:00',
 '2025-06-30 21:00:00', '2025-06-30 22:00:00', '2025-06-30 23:00:00']
Length: 24, dtype: datetime64[s]'], dtype='datetime64[s]', freq=None)

In [ ]:
# Load metrics.npz
metrics_data = np.load(results_dir / "metrics.npz")

print(f"Available metrics: {metrics_data.files}")
for name in metrics_data.files:
    arr = metrics_data[name]
    print(f"\n{name}:")
    print(f"  Shape: {arr.shape}  (num_series, num_windows, num_variates)")
    print(f"  Mean: {np.nanmean(arr):.4f}")
    print(f"  Min:  {np.nanmin(arr):.4f}")
    print(f"  Max:  {np.nanmax(arr):.4f}")